# DFM-LC vs. DFM-MC 15-Day Forecast Animation

Run both forecast models for 60 six-hour steps, save the forecasts, and create a T500 animation comparing DFM-LC and DFM-MC.

In [ ]:
from pathlib import Path
from collections import OrderedDict
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import yaml
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

ROOT = Path.cwd().resolve()
if not (ROOT / 'configs').exists():
    ROOT = (ROOT / '..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dfm_networks.transformer import LGUnet_all

DATA = ROOT / 'data' / 'dfm_lc_mc_case.npz'
LC_CHECKPOINT = ROOT / 'checkpoints' / 'dfm_lc.pth'
MC_CHECKPOINT = ROOT / 'checkpoints' / 'dfm_mc.pth'
CONFIG = ROOT / 'configs' / 'DFM-MC.yaml'
RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
FORECAST_NPZ = RESULTS / 'dfm_lc_mc_15day_forecasts.npz'
ANIMATION_GIF = RESULTS / 'dfm_lc_mc_t500_15day.gif'

for path in [DATA, LC_CHECKPOINT, MC_CHECKPOINT, CONFIG]:
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
case = np.load(DATA, allow_pickle=True)
input_state = case['era5_initial'].astype('float32')
mean = case['mean'].astype('float32')[None, :, None, None]
std = case['std'].astype('float32')[None, :, None, None]
channel_names = [str(x) for x in case['channel_names']]
lead_steps = int(case['lead_steps'])  # 15 days at 6-hour steps = 60

mean_t = torch.from_numpy(mean).to(device)
std_t = torch.from_numpy(std).to(device)

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)
network_params = dict(cfg['model']['network_params'])
network_params['use_checkpoint'] = False

print('lead_steps:', lead_steps)
print('input:', input_state.shape)

In [ ]:
def load_model(checkpoint_path):
    model = LGUnet_all(**network_params)
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint
    clean_state = OrderedDict()
    for key, value in state.items():
        name = key[7:] if key.startswith('module.') else key
        name = name[10:] if name.startswith('_orig_mod.') else name
        if 'logvar' not in name:
            clean_state[name] = value
    missing, unexpected = model.load_state_dict(clean_state, strict=False)
    if missing or unexpected:
        print('missing:', missing)
        print('unexpected:', unexpected)
    model.to(device).eval()
    return model

@torch.no_grad()
def rollout(checkpoint_path):
    model = load_model(checkpoint_path)
    x = torch.from_numpy(input_state).to(device)
    x = (x - mean_t) / std_t
    frames = []
    for step in range(lead_steps):
        x = model(x)[:, :len(channel_names)]
        frames.append((x * std_t + mean_t).cpu().numpy()[0])
        # print(f'{checkpoint_path.name}: step {step + 1}/{lead_steps}')
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return np.stack(frames, axis=0).astype('float32')

In [ ]:
dfm_lc = rollout(LC_CHECKPOINT)
dfm_mc = rollout(MC_CHECKPOINT)

np.savez_compressed(
    FORECAST_NPZ,
    dfm_lc=dfm_lc,
    dfm_mc=dfm_mc,
    era5_initial=input_state,
    channel_names=np.asarray(channel_names, dtype=object),
    lead_steps=np.int32(lead_steps),
)
print('saved:', FORECAST_NPZ)

In [ ]:
channel = 't_500'
idx = channel_names.index(channel)
plot_lc = dfm_lc[:, idx]
plot_mc = dfm_mc[:, idx]
vmin, vmax = np.percentile(np.stack([plot_lc, plot_mc]), [1, 99])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={'wspace': 0.08})
im_lc = axes[0].imshow(plot_lc[0], vmin=vmin, vmax=vmax, cmap='viridis')
im_mc = axes[1].imshow(plot_mc[0], vmin=vmin, vmax=vmax, cmap='viridis')
axes[0].set_title('DFM-LC')
axes[1].set_title('DFM-MC')
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im_mc, ax=axes, shrink=0.75, location='right', pad=0.02)

def update(frame):
    im_lc.set_array(plot_lc[frame])
    im_mc.set_array(plot_mc[frame])
    fig.suptitle(f'{channel}, lead {(frame + 1) * 6} h')
    return im_lc, im_mc

ani = FuncAnimation(fig, update, frames=lead_steps, interval=250, blit=False)
ani.save(ANIMATION_GIF, writer='pillow', fps=4)
print('saved:', ANIMATION_GIF)
plt.close(fig)
HTML(ani.to_jshtml())